# Leakage-Safe Features, Models And Hyperparameter Tuning

Target month `t` uses demand history only through `t-1`. Planned production/BOM requirement for `t` is allowed only in causal models because it is assumed available at forecast creation time.

The protocol uses six tuning months, six independent champion-selection months, and twelve untouched rolling test origins. Random splitting is not used.


In [1]:
from pathlib import Path
import json
import pandas as pd
try:
    from IPython.display import Image, display
except ImportError:
    class Image:
        def __init__(self, filename): self.filename = filename
        def __repr__(self): return f"Image({self.filename})"
    def display(value): print(value)

ROOT = Path.cwd()
if ROOT.name != "v8_controlled_synthetic_validation":
    candidate = ROOT / "Ai miroservices/modeling/v8_controlled_synthetic_validation"
    ROOT = candidate if candidate.exists() else ROOT
OUT = ROOT / "outputs"
DATA = OUT / "data"
PLOTS = OUT / "plots"
EVAL = OUT / "evaluator"
summary = json.loads((OUT / "run_summary.json").read_text())


In [2]:
trials = pd.read_csv(OUT / "hyperparameter_trials.csv")
trials.head(15).round(4)


,trial_id,num_leaves,learning_rate,min_child_samples,reg_lambda,n_estimators,validation_WAPE,validation_RMSE
0,15,31,0.050,30,0.5,320,0.0882,1242.0790
1,13,31,0.050,12,0.5,320,0.0883,1242.5484
2,16,31,0.050,30,1.5,320,0.0888,1247.7885
3,14,31,0.050,12,1.5,320,0.0892,1268.7038
4,6,16,0.050,12,1.5,320,0.0897,1262.3562
5,5,16,0.050,12,0.5,320,0.0900,1266.4661
6,9,31,0.025,12,0.5,320,0.0900,1257.4028
7,10,31,0.025,12,1.5,320,0.0901,1265.2881
8,7,16,0.050,30,0.5,320,0.0904,1285.9410
9,8,16,0.050,30,1.5,320,0.0905,1278.9377


In [3]:
selection = pd.read_csv(OUT / "selection_leaderboard.csv")
selection.assign(WAPE_pct=100*selection.WAPE, Bias_pct=100*selection.Bias).round(2)


,model,rows,materials,WAPE,MAE,RMSE,Bias,under_forecast_rate,shock_WAPE,WAPE_pct,Bias_pct
0,extra_trees_causal,720,120,0.08,698.34,1456.85,-0.01,0.47,0.09,8.11,-1.00
1,random_forest_causal,720,120,0.08,705.46,1381.32,-0.00,0.46,0.09,8.19,-0.23
2,lightgbm_causal_tweedie,720,120,0.08,707.64,1408.31,-0.01,0.46,0.09,8.22,-0.50
3,bom_plan,720,120,0.08,713.07,1370.67,0.00,0.45,0.09,8.28,0.02
4,lightgbm_causal_ratio,720,120,0.09,756.51,1563.92,-0.01,0.49,0.10,8.79,-0.74
5,lightgbm_direct_ratio,720,120,0.10,845.34,1649.76,-0.02,0.52,0.10,9.82,-2.08
6,holt_damped,720,120,0.10,896.30,1623.36,-0.04,0.58,0.13,10.41,-3.88
7,moving_average_6,720,120,0.13,1152.66,2150.19,-0.01,0.49,0.13,13.39,-0.74
8,seasonal_naive,720,120,0.14,1226.69,2580.11,-0.09,0.62,0.14,14.25,-9.07
9,croston_sba,720,120,0.15,1300.21,2467.41,-0.11,0.71,0.17,15.10,-10.97


Candidate families include seasonal naive, moving average, Croston/SBA, damped Holt/ETS, deterministic BOM-plan, Ridge, Elastic Net, Random Forest, Extra Trees, direct LightGBM, causal LightGBM and Tweedie LightGBM.
